In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Fixed-direction channel attribution

Select a GPU runtime and **Run all**. Uses the fixed BASELINE, COMPOSITE and WORST_ONLY rasters from 20261002T131059238475Z and the same terminal steps and raw420 observations from 20261002T053509124852Z. All input locations are prefilled. The two arms remain matched to their original objectives; neither endpoint is selected or updated.

Only two native encoder VJPs are computed, sequentially at the identical saved baseline raw420 received RGB. COMPOSITE uses the original global hinge plus mean of all 173 pair hinges; WORST_ONLY retains the full-family global hinge. Each full normalized replay is compared with the saved baseline. Cotangents are saved in 16-frame chunks (at most about 32 MB each) so later download and independent dot-product verification are practical. Existing causal checkpoint storage and dependency repair are reused, with no GPU-name or exact-environment gate.

For each matched arm, report five signed loss-change quantities:

1. Saved terminal gradient dotted with actual saved terminal displacement, including float32 step rounding.
2. New baseline raw420 RGB cotangent dotted with saved clamped FLOAT RGB displacement.
3. The same cotangent dotted with saved RGB8/255 displacement.
4. The same cotangent dotted with actual saved raw420 received RGB displacement.
5. Actual objective change, recomputed from saved raw420 normalized tensors over all 173 wrong classes.

A negative prediction means objective decrease. Consecutive differences form a telescoping decomposition of prediction error. The first difference also includes cross-run gradient replay differences; the last includes finite encoder response and piecewise loss effects. All intermediate cotangents belong to the raw420 baseline, not to FLOAT/RGB8 objectives. This is fixed-direction first-order attribution, not proof of a unique physical cause or a new gradient method. Native FLOAT includes the unchanged [0,1] clamp.

The existing large FLOAT files are read directly from mounted Drive. Their quantization to the saved RGB8 rasters and equality to the old channel inputs are checked on CPU and reported. No raster is regenerated. No new decode, decoder VJP, writer step, color conversion, receiver search or payload read is performed. Three old raw420 observations, six old blind readouts and twelve message evaluations remain references. Native encoder replay mismatches are reported, without discarding either fixed arm. Failed and missing rows stay in the two-arm denominator.

Static/CPU tests cover arithmetic, saved-input plumbing, fixed denominators, replay reporting, chunk persistence and failure handling. The two full pretrained encoder VJPs are left to user execution. This diagnostic does not establish MP4 robustness, unknown-phase synchronization, path-aligned payload closure, population FPR or generation-time trajectory embedding.

Results: MyDrive/Video-WM/Zero-Mean-Channel-Direction-V1/<timestamp>/fixed_reference/result.json.


In [ ]:
SOURCE_SHA = '23a6c13bb13df7383d942a27d37c8278aa7baa9d'
from pathlib import Path
import datetime,json,os,subprocess,sys,traceback
STAMP=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=Path('/content/drive/MyDrive/Video-WM/Zero-Mean-Channel-Direction-V1')/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
REPO=Path('/content/SC-SSTW-CHANNEL-DIRECTION-'+STAMP)
RUN_OUTPUT=OUTPUT/'fixed_reference'
PYTHON=sys.executable
INPUT_ROOT=Path('/content/drive/MyDrive/Video-WM/Zero-Mean-Fixed-Layer-Forward-V1/20261002T131059238475Z/fixed_reference')
REFERENCE_ROOT=Path('/content/drive/MyDrive/Video-WM/Zero-Mean-Raw420-Objective-Ablation-V1/20261002T053509124852Z/fixed_reference')
setup=dict(status='SETUP_STARTED',source_sha=SOURCE_SHA,fixed_denominator={'encoder_vjps': 2, 'replay_normalized': 2, 'rgb_cotangents': 2, 'attributions': 2, 'updates': 0, 'new_path_reads': 0, 'new_payload_reads': 0, 'reference_observations': 3, 'reference_raw': 6, 'reference_message_evaluations': 12, 'fixed_terminals': 3, 'float_rasters': 3, 'rgb8_rasters': 3, 'raw420_rgb_rasters': 3},observations={o:{'status':'NOT_RUN_SETUP'} for o in []})
def write_json(path,value):
    temp=path.with_suffix(path.suffix+'.tmp');temp.write_text(json.dumps(value,indent=2)+'\n');os.replace(temp,path)
def failed(stage,exc):
    setup.update(status='SETUP_FAILED',stage=stage,error=f'{type(exc).__name__}: {exc}')
    write_json(OUTPUT/'setup_receipt.json',setup)
    write_json(OUTPUT/'setup_failure.json',dict(stage=stage,error=str(exc),traceback=traceback.format_exc()))
def logged(command,stage,cwd=None,check=True):
    try:
        with (OUTPUT/'execution.log').open('a') as log:
            log.write('COMMAND '+repr(command)+'\n');log.flush()
            child=subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            for line in child.stdout:print(line,end='',flush=True);log.write(line);log.flush()
            code=child.wait();log.write('EXIT '+str(code)+'\n')
        if check and code:raise subprocess.CalledProcessError(code,command)
        return code
    except Exception as exc:failed(stage,exc);raise
write_json(OUTPUT/'setup_receipt.json',setup)
print('Full writer terminal:',INPUT_ROOT,'Saved references:',REFERENCE_ROOT,'Output:',OUTPUT)
if SOURCE_SHA is None:
    error=RuntimeError('Unpublished draft: bind verified published source SHA before Run all')
    failed('SOURCE_BINDING',error);raise error
TERMINAL_ROOT=Path('/content/drive/MyDrive/Video-WM/Zero-Mean-Raw420-Margin-Continue-V2/20261002T041230450908Z/fixed_reference')


In [ ]:
try:
    logged(['git','clone','--filter=blob:none','https://github.com/RICHAAARC/SC-SSTW.git',str(REPO)],'SOURCE_CLONE')
    logged(['git','-C',str(REPO),'fetch','origin',SOURCE_SHA],'SOURCE_FETCH')
    logged(['git','-C',str(REPO),'checkout','--detach',SOURCE_SHA],'SOURCE_CHECKOUT')
    actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True)
    if actual!=SOURCE_SHA or dirty:raise RuntimeError('source SHA/clean checkout mismatch')
    write_json(OUTPUT/'source_receipt.json',dict(source_sha=actual,clean=True))
    torch_ok=logged([PYTHON,'-c','import torch,torchvision; print(torch.__version__,torchvision.__version__)'],'TORCH_PROBE',check=False)==0
    if not torch_ok:
        logged([PYTHON,'-m','pip','install','--upgrade','torch','torchvision'],'TORCH_REPAIR')
    logged([PYTHON,'-m','pip','install','-r',str(REPO/'experiments/wan_state_clock/requirements-grow-video-reference.txt')],'DEPENDENCIES')
    dependency_code=logged([PYTHON,'-m','pip','check'],'DEPENDENCY_REPORT',check=False)
    setup['dependency_check_returncode']=dependency_code
    if dependency_code:print('Metadata conflicts retained; actual imports determine availability.')
    (OUTPUT/'environment_freeze.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
    import_code='import json,torch; from diffusers import AutoencoderKLWan; from experiments.wan_state_clock.zero_mean_c1_yuv444_no_h264_run import environment; r=environment(); r.update(cuda_available=torch.cuda.is_available()); print(json.dumps(r))'
    env=json.loads(subprocess.check_output([PYTHON,'-c',import_code],cwd=REPO,text=True))
    write_json(OUTPUT/'environment_receipt.json',env)
    if not env['cuda_available']:print('CUDA unavailable: CPU path selected; much slower.')
    setup.update(status='SETUP_COMPLETE',python_executable=PYTHON,environment_strategy='current_python_fresh_children')
    write_json(OUTPUT/'setup_receipt.json',setup)
except Exception as exc:failed('SOURCE_OR_ENVIRONMENT',exc);raise


In [ ]:
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN')
    if token:os.environ['HF_TOKEN']=token
except Exception:
    pass
finally:
    token=None
command=[PYTHON,'-u','-m','experiments.wan_state_clock.zero_mean_channel_direction_run','--output',str(RUN_OUTPUT),'--input-root',str(INPUT_ROOT),'--reference-root',str(REFERENCE_ROOT),'--terminal-root',str(TERMINAL_ROOT)]
try:
    returncode=logged(command,'FIXED_RUN',cwd=REPO,check=False)
    RESULT_PATH=RUN_OUTPUT/'result.json'
    write_json(OUTPUT/'execution_receipt.json',dict(command=command,returncode=returncode,result_path=str(RESULT_PATH),result_exists=RESULT_PATH.is_file()))
    if not RESULT_PATH.is_file():raise RuntimeError('No runner result; full setup denominator is retained')
    setup.update(status='SUPERSEDED_BY_RESULT',result_path=str(RESULT_PATH));write_json(OUTPUT/'setup_receipt.json',setup)
except Exception as exc:failed('RUNNER',exc);raise


In [ ]:
result=json.loads(RESULT_PATH.read_text())
if result['source_sha']!=SOURCE_SHA or result['fixed_denominator']!=setup['fixed_denominator']:raise RuntimeError('result source/denominator mismatch')
print('Status:',result['status'],'Counts:',result.get('counts'))
print('Calls:',result['calls'])
print('Raster alignment:',result.get('raster_alignment'))
for name,row in result['input_replay'].items():print('Input replay:',name,row)
for name,row in result['encoder_replays'].items():
    print('Encoder replay:',name,row['status'],'max error=',row.get('normalized_max_error'),'RMSE=',row.get('normalized_rmse'),'cotangent chunks=',len(row.get('cotangent_parts',[])))
for name,row in result['attributions'].items():
    print('Attribution:',name,row['status'],'objective=',row.get('objective'))
    print('Predictions:',{k:row.get(k) for k in ('stored_gradient_dot_step','stored_step_prediction_recomputed','terminal_prediction','actual_objective_change','actual_terminal_change_l2')})
    print('RGB dots:',{k:row.get(k,{}).get('total') for k in ('FLOAT','RGB8','RAW420')})
    print('Differences:',row.get('differences'),'telescoping residual=',row.get('telescoping_residual'))
for name,row in result['reference_observations'].items():
    print('Reference:',name,row['posthoc'],'messages=',row['messages'])
print('Resources:',result['resources'])
print('Failures:',result['failures'])
print(result['evidence_ceiling'])
print('Retained result:',RESULT_PATH)
